In [ ]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from pathlib import Path

import kagglehub
import pandas as pd



In [ ]:

path = kagglehub.competition_download("titanic")

train = pd.read_csv(Path(path) / "train.csv")
test = pd.read_csv(Path(path) / "test.csv")

# Combine both files so families and shared tickets link across them.
# Test passengers have no Survived value, so it stays empty for them.
data = pd.concat([train, test], ignore_index=True, sort=False)
n_train = len(train)

data["Surname"] = data["Name"].str.split(",").str[0].str.strip()
data["Title"] = data["Name"].str.extract(r",\s*([^.]+)\.")[0].str.strip()

# Woman or boy: female, or carrying the title "Master"
data["IsWomanOrBoy"] = (data["Sex"] == "female") | (data["Title"] == "Master")

# Adult man: everyone else
data["IsAdultMan"] = ~data["IsWomanOrBoy"]

print("Train:", n_train, "| Test:", len(data) - n_train)
print(data.iloc[:n_train].groupby("IsWomanOrBoy")["Survived"].agg(["size", "mean"]).round(3))

Train: 891 | Test: 418
              size   mean
IsWomanOrBoy             
False          537  0.160
True           354  0.723


In [2]:
# ----- Step 2: find travel groups -----

data["FamilySize"] = data["SibSp"] + data["Parch"] + 1

# Each passenger starts in their own group.
parent = list(range(len(data)))


def find_group(passenger):
    while parent[passenger] != passenger:
        parent[passenger] = parent[parent[passenger]]
        passenger = parent[passenger]
    return passenger


def link_passengers(passengers):
    first = find_group(passengers[0])
    for passenger in passengers[1:]:
        parent[find_group(passenger)] = first


# Link 1: passengers on the same ticket
for passengers in data.groupby("Ticket").groups.values():
    link_passengers(list(passengers))

# Link 2: relatives on different tickets
family = data[data["FamilySize"] > 1]
family_keys = ["Surname", "Pclass", "Embarked", "FamilySize"]
for passengers in family.groupby(family_keys).groups.values():
    link_passengers(list(passengers))

data["Group"] = [find_group(passenger) for passenger in range(len(data))]

group_sizes = data["Group"].value_counts()
print("Travel groups:", len(group_sizes))
print("Passengers in shared groups:", group_sizes[group_sizes > 1].sum())

largest = group_sizes.head(5).index
for group in largest:
    members = data[data["Group"] == group]
    print(f"Size {len(members)}: {sorted(members['Surname'].unique())[:5]}")

Travel groups: 905
Passengers in shared groups: 632
Size 11: ['Sage']
Size 9: ['Andersson']
Size 8: ['Goodwin']
Size 8: ['Bing', 'Chip', 'Foo', 'Hee', 'Lam']
Size 7: ['Asplund']


In [4]:
# ----- Step 3: build a fair test -----

labels = train["Survived"].to_numpy()
splitter = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)


def evaluate(name, predict):
    scores = []

    for fit_rows, val_rows in splitter.split(labels, labels):
        # Hide the answers for the passengers being tested.
        visible = data.copy()
        visible.loc[val_rows, "Survived"] = np.nan

        predictions = predict(visible, fit_rows, val_rows)
        scores.append((predictions == labels[val_rows]).mean())

    scores = np.array(scores)
    print(f"{name}: {scores.mean():.1%} (fold SD {scores.std(ddof=1):.1%})")
    return scores


def gender_rule(visible, fit_rows, predict_rows):
    return (visible.loc[predict_rows, "Sex"] == "female").astype(int).to_numpy()


gender_scores = evaluate("Gender rule", gender_rule)


def cheating_model(visible, fit_rows, predict_rows):
    return visible.loc[predict_rows, "Survived"].fillna(0).astype(int).to_numpy()


cheating_scores = evaluate("Safety check", cheating_model)
print(f"Share who died: {1 - labels.mean():.1%}")

Gender rule: 78.7% (fold SD 2.6%)
Safety check: 61.6% (fold SD 0.2%)
Share who died: 61.6%


In [12]:
# ----- Step 4: the rule-only benchmark -----


def group_rule(visible, fit_rows, predict_rows):
    # Outcomes of women and boys whose results the model may see
    known = visible.loc[fit_rows]
    known = known[known["IsWomanOrBoy"]]
    outcomes = known.groupby("Group")["Survived"].agg(["count", "sum"])

    passengers = visible.loc[predict_rows].join(outcomes, on="Group")
    has_known_members = passengers["count"] > 0
    all_survived = has_known_members & (passengers["sum"] == passengers["count"])
    all_died = has_known_members & (passengers["sum"] == 0)

    is_woman = passengers["Sex"] == "female"
    is_boy = passengers["IsWomanOrBoy"] & ~is_woman

    # Women survive unless their whole known group died.
    # Boys die unless their whole known group survived.
    # Adult men always die.
    survived = (is_woman & ~all_died) | (is_boy & all_survived)
    return survived.astype(int).to_numpy()


rule_scores = evaluate("Group rule", group_rule)

wins = (rule_scores > gender_scores).sum()
print(f"Beat the gender rule in {wins} of {len(rule_scores)} folds")


OUTPUT_DIR = Path(r"C:\Users\user\Desktop\KaggleTitanic")


def save_submission(predict, filename):
    fit_rows = np.arange(n_train)
    predict_rows = np.arange(n_train, len(data))
    predictions = predict(data, fit_rows, predict_rows)

    submission = pd.DataFrame({
        "PassengerId": test["PassengerId"],
        "Survived": predictions,
    })
    assert submission.shape == (418, 2)
    assert set(submission["Survived"]) <= {0, 1}

    output_path = OUTPUT_DIR / filename
    submission.to_csv(output_path, index=False)
    print(f"Saved {output_path}: {submission['Survived'].sum()} predicted survivors")


save_submission(group_rule, "submission_group_rule.csv")

Group rule: 83.5% (fold SD 2.9%)
Beat the gender rule in 25 of 25 folds
Saved C:\Users\user\Desktop\KaggleTitanic\submission_group_rule.csv: 152 predicted survivors


In [13]:
print("Saved in:", Path("submission_group_rule.csv").resolve())

Saved in: C:\Users\user\Downloads\submission_group_rule.csv


In [14]:
# ----- Step 5: test one extra rule -----

data["FarePerPerson"] = data["Fare"] / data.groupby("Ticket")["Ticket"].transform("size")


def group_rule_plus(visible, fit_rows, predict_rows):
    predictions = group_rule(visible, fit_rows, predict_rows).copy()

    known = visible.loc[fit_rows]
    known = known[known["IsWomanOrBoy"]]
    passengers = visible.loc[predict_rows]

    # 3rd-class women from Southampton with no known group members,
    # who paid at least 8 per person, are predicted to die.
    risky = (
        (passengers["Sex"] == "female")
        & ~passengers["Group"].isin(known["Group"])
        & (passengers["Pclass"] == 3)
        & (passengers["Embarked"] == "S")
        & (passengers["FarePerPerson"] >= 8)
    )

    predictions[risky.to_numpy()] = 0
    return predictions


plus_scores = evaluate("Group rule plus", group_rule_plus)

better = (plus_scores > rule_scores).sum()
worse = (plus_scores < rule_scores).sum()
print(f"Compared with the group rule: better in {better} folds, worse in {worse}")

save_submission(group_rule_plus, "submission_group_rule_plus.csv")

Group rule plus: 84.3% (fold SD 2.6%)
Compared with the group rule: better in 13 folds, worse in 4
Saved C:\Users\user\Desktop\KaggleTitanic\submission_group_rule_plus.csv: 141 predicted survivors


In [15]:
# ----- Step 7: add relatives on nearly matching tickets -----

near = data.assign(TicketStem=data["Ticket"].str.replace(r"\D", "", regex=True).str[:-1])
near_keys = ["Surname", "Pclass", "Embarked", "TicketStem"]

for passengers in near.groupby(near_keys).groups.values():
    link_passengers(list(passengers))

near_groups = [find_group(passenger) for passenger in range(len(data))]
print("Travel groups:", len(set(near_groups)))


def group_rule_near(visible, fit_rows, predict_rows):
    return group_rule(visible.assign(Group=near_groups), fit_rows, predict_rows)


near_scores = evaluate("Group rule near", group_rule_near)

better = (near_scores > rule_scores).sum()
worse = (near_scores < rule_scores).sum()
print(f"Compared with the group rule: better in {better} folds, worse in {worse}")

save_submission(group_rule_near, "submission_near_ticket.csv")

Travel groups: 889
Group rule near: 83.6% (fold SD 2.8%)
Compared with the group rule: better in 6 folds, worse in 0
Saved C:\Users\user\Desktop\KaggleTitanic\submission_near_ticket.csv: 151 predicted survivors
